# FASE 2: Diseño de Componentes Modulares de Preprocesamiento

In [ ]:
class Winsorizer(BaseEstimator, TransformerMixin):
  """
  Tratamiento de atípicos

  Parámetros
  ----------
  BaseEstimator : Clase base para estimadores en scikit-learn.
  TransformerMixin : Clase base para transformadores en scikit-learn.

  Atributos
  ---------
  columns_ : array-like
    Nombres de las columnas a transformar.
  limits : tuple
    % de los extremos a descartar
  """
  def __init__(self, limits=(0.05, 0.05)):
    self.limits = limits

  def fit(self, X, y=None):
    # Guardar nombres si es DataFrame, si no generar nombres genéricos
    if isinstance(X, pd.DataFrame):
      self.columns_ = X.columns
    else:
      self.columns_ = np.arange(X.shape[1])
    return self

  def transform(self, X):
    X = pd.DataFrame(X, columns=self.columns_)
    for col in self.columns_:
      lower = X[col].quantile(self.limits[0])
      upper = X[col].quantile(1 - self.limits[1])
      X = X.astype("float64")
      X[col] = np.clip(X[col], lower, upper)
    return X

  def get_feature_names_out(self, input_features=None):
    if input_features is None:
      return np.array(self.columns_)
    else:
      return np.array(input_features)

In [ ]:
def tratar_duplicados(X : pd.DataFrame, drop = True):
  """
  Tratamiento de duplicados

  Parámetros
  ----------
  X : DataFrame
    Conjunto de datos.
  drop : bool
    Si se deben eliminar los duplicados.

  Retorna
  -------
  DataFrame
    Conjunto de datos sin duplicados.
  """
  return X.drop_duplicates() if drop else X

In [ ]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
  """
  Eliminación de variables correlacionadas

  Parámetros
  ----------
  BaseEstimator : Clase base para estimadores en scikit-learn.
  TransformerMixin : Clase base para transformadores en scikit-learn.

  Atributos
  ---------
  columns_to_drop_ : array-like
    Nombres de las columnas a eliminar.
  threshold : float
    Umbral de correlación.
  Returns
  -------
  DataFrame
    Conjunto de datos sin variables correlacionadas.
  """
  def __init__(self, threshold=0.9):
    self.threshold = threshold
    self.columns_to_drop_ = None

  def fit(self, X, y=None):
    X_df = pd.DataFrame(X)

    corr_matrix = X_df.corr().abs()
    upper = corr_matrix.where(
      np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    self.columns_to_drop_ = [
        col for col in upper.columns if any(upper[col] > self.threshold)
    ]

    return self

  def transform(self, X):
    X_df = pd.DataFrame(X)
    X_filtered = X_df.drop(columns=self.columns_to_drop_, errors="ignore")
    return X_filtered.values

In [ ]:
class DataFrameConverter(BaseEstimator, TransformerMixin):
  """
  Convierte un array en un DataFrame

  Parámetros
  ----------
  BaseEstimator : Clase base para estimadores en scikit-learn.
  TransformerMixin : Clase base para transformadores en scikit-learn.

  Atributos
  ---------
  feature_names_ : array-like
    Nombres de las columnas.
  Returns
  -------
  DataFrame
    Conjunto de datos con nombres de columnas.
  """
  def __init__(self, preprocessor):
    self.preprocessor = preprocessor
    self.feature_names_ = None

  def fit(self, X, y=None):
    # Obtener nombres después de fit del preprocessor
    self.feature_names_ = self.preprocessor.get_feature_names_out()
    return self

  def transform(self, X):
    return pd.DataFrame(X, columns=self.feature_names_)

Encapsular las transformaciones en clases es una exigencia arquitectónica porque el motor de Scikit-Learn requiere objetos equipados con los métodos .fit() y .transform() para automatizar un Pipeline. El beneficio técnico más crítico es que las clases retienen "memoria" (estado), lo que les permite calcular parámetros —como percentiles o umbrales de correlación— exclusivamente con los datos de entrenamiento y aplicarlos de forma idéntica al set de prueba, previniendo la grave falla del Data Leakage. En contraste, las funciones simples se reservan únicamente para acciones que no requieren memoria matemática, como eliminar filas duplicadas.

# Construcción de Pipelines y Modelamiento Supervisado Base (clasificación)

In [ ]:
"""
FASE 3: CONFIGURACIÓN DE SUB-PIPELINES ESPECIALIZADOS

PROPÓSITO:
Definir los bloques secuenciales de transformación modular (Pipeline) encargados
de procesar de forma diferenciada las variables según su naturaleza estadística
(numéricas, nominales y ordinales).

IMPORTANCIA PARA EL ENCARGO:
Este diseño es el núcleo del preprocesamiento industrial exigido por la pauta.
Asegura que operaciones críticas como el cálculo de percentiles del 'Winsorizer',
los promedios del 'SimpleImputer' y los límites del 'MinMaxScaler' se ejecuten
estrictamente de manera interna. Esto congela los aprendizajes matemáticos en el
set de entrenamiento y evita la filtración de datos (Data Leakage) hacia el set de
prueba. Además, prepara la geometría de los datos para algoritmos altamente
sensibles a la escala y la codificación, como la Regresión Logística y la SVM.
"""

# 1. Pipeline para variables numéricas (Limpieza + Imputación + Escalado)
pipeline_numerico = Pipeline(
    steps=[
        ("winsorizer", Winsorizer(limits=(0.05, 0.05))),
        ("imputacion", SimpleImputer(strategy="mean")),
        ("escalado", MinMaxScaler())
    ]
)

# 2. Pipeline para variables categóricas Nominales (Sin orden -> OneHot)
pipeline_nominal = Pipeline(
    steps=[
        ("imputacion", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(sparse_output=False, handle_unknown="ignore"))
    ]
)

# 3. Pipeline para variables categóricas Ordinales (Con orden -> OrdinalEncoder)
pipeline_ordinal = Pipeline(
    steps=[
        ("imputacion", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(categories=[
            ['Bajo', 'Medio', 'Alto'],          # Orden jerárquico para 'uso_app'
            ['Basico', 'Estandar', 'Premium'] # Orden jerárquico para 'tipo_plan'
        ]))
    ]
)

In [ ]:
"""
PROPÓSITO:
Construir el preprocesador maestro (ColumnTransformer) mediante la integración
de tres ramas de transformación independientes: numérica (pipeline_numerico),
nominal (pipeline_nominal) y ordinal (pipeline_ordinal), mapeadas directamente
a sus respectivas listas de características analíticas.

IMPORTANCIA PARA EL ENCARGO:
Este bloque modular es el núcleo de la ingeniería de características del proyecto.
Permite aplicar transformaciones estadísticas diferenciadas en un único paso
automatizado según la naturaleza de cada variable (escalado para numéricas,
codificación binaria para nominales y mapeo jerárquico para ordinales). Al
centralizar el flujo, se garantiza que el cálculo de parámetros matemáticos ocurra
estrictamente durante la fase de ajuste (.fit) en el set de entrenamiento, previniendo
de forma absoluta el sesgo por filtración de datos (Data Leakage) hacia el set de prueba
y asegurando la estabilidad geométrica que requieren algoritmos como la SVM y la Regresión.

MODIFICACIÓN CRÍTICA:
Se cambia 'remainder="passthrough"' por 'remainder="drop"'. Esto elimina de forma
automática cualquier columna que no haya sido clasificada explícitamente en nuestras
listas (como identificadores de clientes o fechas en bruto tipo '2021-03-03'),
evitando que ingresen variables de texto que corrompan los cálculos matemáticos del modelo.
"""
preprocessor = ColumnTransformer(
    transformers=[
        ("num", pipeline_numerico, numeric_features),
        ("nom", pipeline_nominal, categorical_nominal_features),
        ("ord", pipeline_ordinal, categorical_ordinal_features)
    ],
    remainder="drop", # <-- Cambiado de "passthrough" a "drop"
    force_int_remainder_cols=False
)

In [ ]:
def evaluar(modelo: BaseEstimator, X_train: np.array, X_test: np.array, y_train: np.array, y_test: np.array):
  """
  Retorna las métricas del modelo

  Parámetros
  ----------
  modelo : BaseEstimator
    Modelo a evaluar.
  X_train : np.array
    Conjunto de datos de entrenamiento.
  X_test : np.array
    Conjunto de datos de prueba.
  y_train : np.array
    Etiquetas de entrenamiento.
  y_test : np.array
    Etiquetas de prueba
  Returns
  -------
  dict
    Diccionario con las métricas del modelo.

  """
  modelo.fit(X_train, y_train)

  y_pred = modelo.predict(X_test)
  y_prob = modelo.predict_proba(X_test)[:,1]

  return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "matriz_confusion": confusion_matrix(y_test, y_pred)
    }

In [ ]:
"""
PROCESO: AISLAMIENTO DE VARIABLES Y PARTICIÓN ESTRATIFICADA

PROPÓSITO:
Preparar el conjunto de datos para el entrenamiento del modelo, eliminando
ruido estructural y garantizando una partición estadísticamente equilibrada.

JUSTIFICACIÓN TÉCNICA:
1. Limpieza de Variables (Feature Selection):
   - Se elimina 'id_cliente' para evitar el sobreajuste (el modelo no debe
     memorizar identificadores).
   - Se elimina 'codigo_postal' por ser una variable de alta cardinalidad
     que introduciría ruido dimensional innecesario mediante el OneHotEncoder.

2. Estratificación (stratify=y_clasificacion):
   En problemas de Churn con desbalance de clases, esta técnica es obligatoria.
   Garantiza que la tasa de abandono sea idéntica en Entrenamiento y Prueba,
   asegurando que el modelo aprenda de una distribución que represente la
   realidad del negocio.

IMPLICANCIA DE NEGOCIO:
Al asegurar que tanto el set de entrenamiento como el de prueba contengan
la misma proporción de clientes en riesgo, las métricas de evaluación
(F1-Score, Recall) serán estadísticamente válidas, permitiendo tomar
decisiones de inversión en retención basadas en datos reales y no en
distribuciones sesgadas.
"""

# 1. SEPARACIÓN DE CARACTERÍSTICAS Y TARGET
# Eliminamos identificadores (id_cliente) y variables de ruido (codigo_postal)
X_clasificacion = df.drop(columns=['id_cliente', 'codigo_postal', 'abandono'])
y_clasificacion = df['abandono']

# 2. PARTICIÓN DE LOS DATOS (TRAIN/TEST SPLIT)
# La estratificación asegura la representatividad de la clase minoritaria (abandono)
X_train, X_test, y_train, y_test = train_test_split(
    X_clasificacion,
    y_clasificacion,
    test_size=0.20,
    random_state=SEED,
    stratify=y_clasificacion
)

# 3. VERIFICACIÓN DE INTEGRIDAD
print(f"Dimensiones de entrenamiento - X_train: {X_train.shape} | y_train: {y_train.shape}")
print("=== PARTICIÓN DE CLASIFICACIÓN (CHURN) COMPLETADA ===")
print(f"Conjunto de Entrenamiento (Train) : {X_train.shape[0]} filas")
print(f"Conjunto de Prueba (Test)         : {X_test.shape[0]} filas")
print(f"Proporción de Churn en Train      : {y_train.mean():.2%}")
print(f"Proporción de Churn en Test       : {y_test.mean():.2%}\n")

Dimensiones de entrenamiento - X_train: (16000, 19) | y_train: (16000,)
=== PARTICIÓN DE CLASIFICACIÓN (CHURN) COMPLETADA ===
Conjunto de Entrenamiento (Train) : 16000 filas
Conjunto de Prueba (Test)         : 4000 filas
Proporción de Churn en Train      : 39.67%
Proporción de Churn en Test       : 39.67%



Dividimos los datos en dos grupos para evaluar el modelo (80% entrenamiento y 20% prueba):

1. **Grupo de Entrenamiento (80%):** Es el set que el modelo utiliza para aprender. Al haber limpiado los duplicados previamente, nos aseguramos de que el modelo aprenda con registros únicos y fiables, evitando que memorice datos repetidos y logrando un aprendizaje real.
2. **Grupo de Prueba (20%):** Es el "examen final". Son datos que el modelo nunca ha visto, por lo que nos permiten verificar si realmente aprendió a predecir o si solo memorizó la información.

**Punto clave:** Utilizamos una partición "estratificada". Esto garantiza que el porcentaje de clientes que abandonan (aprox. 39%) sea idéntico en ambos grupos. Es fundamental para que la evaluación sea justa, equilibrada y representativa de la realidad del negocio.


## Logistic Regression

In [ ]:
"""
FASE 4: CONFIGURACIÓN DEL PIPELINE DE PRODUCCIÓN (REGRESIÓN LOGÍSTICA)

PROPÓSITO:
Construir y encapsular la arquitectura final de cinco etapas para el modelo de
Regresión Logística, integrando de manera secuencial el paso de duplicados,
el preprocesamiento multivariable, la conversión tabular, el filtrado de
colinealidad y el estimador final.

JUSTIFICACIÓN TÉCNICA DEL PARÁMETRO 'drop=False':
En esta arquitectura se configura la etapa de duplicados con 'drop=False'. Esto se
debe a que eliminar filas físicamente dentro de un Pipeline estándar de Scikit-Learn
durante el ajuste (.fit) provoca un colapso crítico por desajuste de dimensiones
(ValueError), ya que la tubería transforma la matriz X pero no puede modificar en
paralelo el vector objetivo y. Para resolver esto respetando las buenas prácticas,
los datos duplicados se eliminan previamente de forma global sobre el DataFrame antes
del 'train_test_split'. De este modo, este componente se mantiene en el Pipeline
como un eslabón de consistencia arquitectónica, pero sin alterar la geometría de las
muestras durante el entrenamiento.
"""

pipeline_modelo_lr = Pipeline(steps=[
    ("duplicados", FunctionTransformer(tratar_duplicados, kw_args={"drop": False})),
    ("preprocesador", preprocessor),
    ("conversion", DataFrameConverter(preprocessor)),
    ("colinealidad", CorrelationFilter(threshold=0.9)),
    ("modelo", LogisticRegression(max_iter=1000, random_state=SEED))
])


# DecisionTreeClassifier

In [ ]:
"""
PROPÓSITO:
Calcular el impacto real que tiene cada variable predictora en el rendimiento final
del Árbol de Decisión, utilizando el método avanzado de permutación sobre el
conjunto de datos de prueba (unseen data).

IMPORTANCIA PARA EL DESARROLLO DEL ENCARGO:
Este análisis es fundamental para cumplir con los criterios de explicabilidad del
modelo. A diferencia de los indicadores nativos de los árboles
(que se calculan en entrenamiento y suelen beneficiar falsamente a variables muy
específicas), la permutación altera al azar una columna en el set de prueba y mide
cuánto "sufre" o cae la precisión del modelo sin esa información.

Esto nos aporta un valor doble:
1. Justificación Técnica: Nos permite validar si el modelo está tomando decisiones
   basadas en patrones reales o si se está dejando llevar por ruido o variables
   redundantes que pasaron los filtros previos.
2. Valor de Negocio: Transforma los números abstractos en información accionable
   para la empresa, identificando con total certeza cuáles son los verdaderos gatillantes
   que hacen que un cliente decida abandonar el servicio (Churn).
"""



# 1. Ajustar el pipeline del árbol con los datos de entrenamiento
pipeline_modelo_dtc.fit(X_train, y_train)

# 2. Calcular la importancia por permutación en el set de prueba
result = permutation_importance(
    pipeline_modelo_dtc,
    X_test,
    y_test,
    n_repeats=5,
    random_state=29,
    n_jobs=-1
)

# 3. Convertir los resultados en una serie de Pandas indexada por las columnas
importancias = pd.Series(result.importances_mean, index=X_test.columns)

# 4. Mostrar los resultados ordenados de mayor a menor impacto
print("=== IMPORTANCIA DE VARIABLES EN EL SET DE PRUEBA ===")
print(importancias.sort_values(ascending=False))

=== IMPORTANCIA DE VARIABLES EN EL SET DE PRUEBA ===
ultima_compra_dias       0.02550
uso_app                  0.01825
tipo_plan                0.00815
antiguedad_meses         0.00425
canal_registro           0.00355
estado_civil             0.00325
num_productos            0.00320
dia_semana_registro      0.00245
score_crediticio         0.00165
region                   0.00160
edad                     0.00150
deuda_total              0.00125
genero                   0.00070
ingreso_mensual          0.00045
tiene_tarjeta_credito    0.00005
fecha_registro           0.00000
hora_registro            0.00000
frecuencia_compra       -0.00130
gasto_mensual           -0.00140
dtype: float64


+ Valor alto positivo IMPLICA la variable aporta al modelo
+ Cerca de 0 IMPLICA el modelo casi no la usa
+ Negativo IMPLICA al permutarla el modelo mejora (señal de ruido o sobreajuste)

In [ ]:
# Detecta las variables que tienen importancia menor a un umbral
features_sospechosas = importancias[importancias < 0.005].index.tolist()
print(features_sospechosas)

['fecha_registro', 'edad', 'genero', 'region', 'estado_civil', 'ingreso_mensual', 'gasto_mensual', 'deuda_total', 'score_crediticio', 'antiguedad_meses', 'frecuencia_compra', 'num_productos', 'tiene_tarjeta_credito', 'canal_registro', 'dia_semana_registro', 'hora_registro']


In [ ]:
"""
PROPÓSITO:
Crear una copia aislada del dataset para construir un conjunto de características
reducido (X_reducido), eliminando aquellas variables identificadas como redundantes
o de bajo impacto por el análisis de importancia por permutación.

IMPORTANCIA EXPERIMENTAL:
Permite ejecutar un experimento controlado de reducción de dimensionalidad (remoción de ruido)
para comparar el rendimiento del modelo frente al dataset completo. Al utilizar '.copy()',
se evita la modificación por referencia del DataFrame original, protegiendo la integridad
de las matrices globales de entrenamiento y prueba (X_train y X_test) para los demás modelos.
"""
X = df.copy()
X_reducido = X.drop(columns=features_sospechosas)
X_reducido.columns.values

In [ ]:
"""
FASE 4.2: EXPERIMENTO DE REDUCCIÓN DE DIMENSIONALIDAD CON ÁRBOL PODADO

PROPÓSITO:
Evaluar el impacto de la remoción de variables de bajo impacto (features_sospechosas)
sobre el rendimiento de un Árbol de Decisión regularizado con una profundidad
máxima controlada (max_depth=4).

CORRECCIONES CRÍTICAS INCORPORADAS:
1. Resolución de NameError y Data Leakage: Se aísla la columna 'abandono' de la
   matriz X_reducido antes de la partición. Esto soluciona el error de variable
   no definida ('y') y evita que el modelo haga trampa al tener la respuesta
   correcta dentro de las características predictoras.
2. Sincronización de Columnas: Se filtran dinámicamente las listas originales de
   características (numéricas, nominales y ordinales) contrastándolas con las
   columnas vigentes de X_reducido. Esto previene un colapso matemático (KeyError)
   dentro del ColumnTransformer al intentar buscar variables que ya fueron eliminadas.

EXPLICACIÓN DEL PASO A PASO (PARA USUARIOS NO TÉCNICOS):
1. Separación de Datos y Respuestas: Dividimos nuestra tabla reducida en dos partes:
   por un lado, guardamos los datos de comportamiento de los clientes y, por el otro,
   guardamos la respuesta correcta (si abandonó o no). Esto evita que la inteligencia
   artificial haga trampa viendo la solución de antemano.
2. Identificación de Columnas Sobrevivientes: Revisamos cuáles de variables vigentes.
3. Configuración del Preparador: Creamos un organizador automático de limpieza.
4. Construcción de la Tubería Automatizada (Pipeline): Ensamblaje limpio hacia el árbol.
5. División en Grupos de Estudio y Examen: Partición 80/20.
6. Ejecución del Examen: Entrenamiento y prueba.
7. Impresión de Notas Finales: Resultados finales del modelo podado.
"""

# 1. Separación de variables
X_reducido_features = X_reducido.drop(columns=["abandono"])
y_reducido = X_reducido["abandono"]

# 2. Sincronización dinámica de columnas
columnas_vigentes = X_reducido_features.columns
features_num_reducido = [col for col in numeric_features if col in columnas_vigentes]
features_nom_reducido = [col for col in categorical_nominal_features if col in columnas_vigentes]
features_ord_reducido = [col for col in categorical_ordinal_features if col in columnas_vigentes]

# 3. Preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ("num", pipeline_numerico, features_num_reducido),
        ("nom", pipeline_nominal, features_nom_reducido),
        ("ord", pipeline_ordinal, features_ord_reducido)
    ],
    remainder="drop",
    force_int_remainder_cols=False
)

# 4. Pipeline LIMPIO (Sin pasos zombies de duplicados o conversión manual)
pipeline_modelo_dtc = Pipeline(steps=[
    ("preprocesador", preprocessor),
    ("colinealidad", CorrelationFilter(threshold=0.9)),
    ("modelo", DecisionTreeClassifier(max_depth=4, random_state=29))
])

# 5. Partición Estratificada
X_train_sin_sospecha, X_test_sin_sospecha, y_train_sin_sospecha, y_test_sin_sospecha = train_test_split(
    X_reducido_features,
    y_reducido,
    test_size=0.2,
    random_state=SEED,
    stratify=y_reducido
)

# 6. Evaluación
metricas_dtc_ss = evaluar(
    pipeline_modelo_dtc,
    X_train_sin_sospecha,
    X_test_sin_sospecha,
    y_train_sin_sospecha,
    y_test_sin_sospecha
)


# SVM

## Modelo 1

```python
SVC(kernel="linear", probability=True)

In [ ]:
"""
FASE 4.3: CONFIGURACIÓN DEL PIPELINE DE PRODUCCIÓN (SVM - KERNEL LINEAL)

PROPÓSITO:
Construir y encapsular la arquitectura secuencial de cinco etapas para el modelo de
Máquina de Vectores de Soporte (SVM), integrando el procesamiento multivariable,
el filtrado de redundancias y un estimador basado en separación geométrica lineal.

EXPLICACIÓN TÉCNICA DEL KERNEL LINEAL:
El parámetro 'kernel="linear"' le indica al algoritmo que busque la frontera de
decisión utilizando un hiperplano recto (una línea en 2D, un plano en 3D o un
hiperplano plano en dimensiones superiores) que maximice la distancia o "margen"
entre los clientes que abandonan (Churn) y los que se quedan. Se selecciona en esta
fase como un paso metodológico fundamental por tres razones:
1. Simplicidad y Eficiencia: Es computacionalmente rápido y menos propenso al
   sobreentrenamiento (Overfitting) en comparación con kernels complejos.
2. Línea de Base Geométrica: Permite evaluar si las clases del negocio pueden
   separarse de forma directa mediante tendencias lineales antes de intentar
   transformaciones curvas más costosas.
3. Calibración de Probabilidades: El uso de 'probability=True' activa internamente
   el método de Platt, permitiendo que el modelo no solo entregue una etiqueta rígida,
   sino también el porcentaje de certeza de la predicción, requisito clave para
   calcular métricas avanzadas como el ROC-AUC Score.

EXPLICACIÓN DEL PASO A PASO (PARA USUARIOS NO TÉCNICOS):
1. Control de Consistencia: Los datos ingresan a la tubería manteniendo el indicador
   de duplicados en 'False' para proteger la geometría de las filas de entrenamiento.
2. Transformación Automatizada: El preprocesador traduce de forma simultánea los textos
   a números y estandariza los montos de dinero para que todas las variables hablen
   el mismo idioma matemático.
3. Reconstrucción de la Tabla: Convertimos la matriz resultante de vuelta a una tabla
   limpia de Pandas para que los nombres de las columnas sigan siendo legibles.
4. Eliminación de Datos Redundantes: El filtro de colinealidad descarta variables
   que dicen exactamente lo mismo para evitar confundir al algoritmo y hacer el proceso
   más liviano.
5. Trazado de la Frontera Recta: La Máquina de Vectores de Soporte dibuja una línea
   divisoria recta ideal que separa a los clientes estables de los clientes en riesgo
   de fuga, dejando un pasillo de seguridad lo más ancho posible entre ambos grupos.
"""

# 1. Pipeline conservando los pasos originales
pipeline_modelo_svm = Pipeline(steps=[
    ("duplicados", FunctionTransformer(tratar_duplicados, kw_args={"drop": False})),
    ("preprocesador", preprocessor),
    ("conversion", DataFrameConverter(preprocessor)),
    ("colinealidad", CorrelationFilter(threshold=0.9)),
    ("modelo", SVC(kernel="linear", probability=True, random_state=29))
])

# 2. Evaluación usando el DATASET COMPLETO (X_train, y_train originales)
metricas_svm = evaluar(
    pipeline_modelo_svm,
    X_train,
    X_test,
    y_train,
    y_test
)

# 3. Impresión de Métricas
print("=== MÉTRICAS DEL MODELO SVM (KERNEL LINEAL Y DATASET COMPLETO) ===")
print(f"{'Accuracy':<20}: {metricas_svm['accuracy']:.4f}")
print(f"{'Precision':<20}: {metricas_svm['precision']:.4f}")
print(f"{'Recall':<20}: {metricas_svm['recall']:.4f}")
print(f"{'F1 Score':<20}: {metricas_svm['f1']:.4f}")
print(f"{'ROC AUC Score':<20}: {metricas_svm['roc_auc']:.4f}\n")

# 4. Diagnóstico de Overfitting
print("=== DIAGNÓSTICO DE OVERFITTING (SVM) ===")
print(f"{'Score en entrenamiento (Train)':<35}: {pipeline_modelo_svm.score(X_train, y_train):.5f}")
print(f"{'Score en prueba (Test)':<35}: {pipeline_modelo_svm.score(X_test, y_test):.5f}")

## Modelo 2

```python
SVC(kernel="rbf", probability=True)

In [ ]:
"""
FASE 4.4: CONFIGURACIÓN DEL PIPELINE DE PRODUCCIÓN (SVM - KERNEL RBF)

PROPÓSITO:
Construir y encapsular la arquitectura secuencial de cinco etapas para el modelo de
Máquina de Vectores de Soporte (SVM), utilizando un núcleo de Función de Base Radial
(RBF) diseñado para capturar fronteras de decisión altamente complejas y no lineales.

EXPLICACIÓN TÉCNICA DEL KERNEL RBF:
El parámetro 'kernel="rbf"' (Radial Basis Function) aplica el denominado "Kernel Trick"
proyectando matemáticamente el conjunto de datos original hacia un espacio de características
de dimensiones infinitas. En este nuevo espacio transformado, clases que eran totalmente
inseparables de forma lineal se vuelven geométricamente separables mediante un hiperplano.
Esto permite al algoritmo trazar límites de decisión curvos y flexibles en el espacio original,
siendo la herramienta ideal cuando las relaciones de negocio (como el perfil de Churn)
no siguen una tendencia recta o proporcional. Al igual que en la versión lineal,
'probability=True' implementa la calibración de Platt para calcular los porcentajes de certeza
requeridos en la métrica ROC-AUC.

EXPLICACIÓN DEL PASO A PASO (PARA USUARIOS NO TÉCNICOS):
1. Control de Consistencia: Los datos entran a la línea de ensamblaje manteniendo el paso de
   duplicados inactivo ('False') para resguardar la simetría geométrica entre las filas de
   estudio y sus respuestas correspondientes.
2. Transformación Automatizada: El preprocesador unifica los datos traduciendo los textos a
   números y estandarizando los montos de dinero para evitar que las variables con valores
   más grandes confundan al modelo.
3. Reconstrucción de la Tabla: La matriz matemática se convierte nuevamente en una tabla estructurada
   de Pandas para asegurar que las columnas sigan siendo identificables y legibles.
4. Eliminación de Datos Redundantes: El filtro de colinealidad analiza y descarta variables que
   se duplican o dicen exactamente lo mismo, optimizando la velocidad y el rendimiento del algoritmo.
5. Trazado de Fronteras Curvas: A diferencia del modelo anterior que usaba una línea recta, la
   Máquina de Vectores de Soporte con Kernel RBF dibuja zonas y fronteras divisorias curvas,
   adaptándose de manera mucho más orgánica a los comportamientos y patrones complejos de
   los clientes en riesgo de abandonar la empresa.
"""

# 1. Pipeline conservando tu estructura
pipeline_modelo_svm_rbf = Pipeline(steps=[
    ("duplicados", FunctionTransformer(tratar_duplicados, kw_args={"drop": False})),
    ("preprocesador", preprocessor),
    ("conversion", DataFrameConverter(preprocessor)),
    ("colinealidad", CorrelationFilter(threshold=0.9)),
    ("modelo", SVC(kernel="rbf", probability=True, random_state=29))
])

# 2. Evaluación usando el DATASET COMPLETO ORIGINAL (X_train, y_train)
metricas_svm_rbf = evaluar(
    pipeline_modelo_svm_rbf,
    X_train,
    X_test,
    y_train,
    y_test
)

# 3. Impresión de Métricas
print("=== MÉTRICAS DEL MODELO SVM (KERNEL RBF Y DATASET COMPLETO) ===")
print(f"{'Accuracy':<20}: {metricas_svm_rbf['accuracy']:.4f}")
print(f"{'Precision':<20}: {metricas_svm_rbf['precision']:.4f}")
print(f"{'Recall':<20}: {metricas_svm_rbf['recall']:.4f}")
print(f"{'F1 Score':<20}: {metricas_svm_rbf['f1']:.4f}")
print(f"{'ROC AUC Score':<20}: {metricas_svm_rbf['roc_auc']:.4f}\n")

# 4. Diagnóstico de Overfitting
print("=== DIAGNÓSTICO DE OVERFITTING (SVM RBF) ===")
print(f"{'Score del modelo en entrenamiento':<40}: {pipeline_modelo_svm_rbf.score(X_train, y_train):.5f}")
print(f"{'Score del modelo en test':<40}: {pipeline_modelo_svm_rbf.score(X_test, y_test):.5f}")

# Construcción de Pipelines y Modelamiento Supervisado Base (Regresion)

In [ ]:
"""
Aqui se elimina la variable 'gasto_mensual'
porque al ser la variable objetivo no deberia ir
dentro del pipeline. Eventualmente esto genera un problema
al utilizar el modelo, y es que existiran datos tipo 'Nan'
por lo que el modelo fallará. Se corrige mas adelante
"""
numeric_features = [
    'edad',
    'ingreso_mensual',
    'deuda_total',
    'score_crediticio',
    'antiguedad_meses',
    'frecuencia_compra',
    'ultima_compra_dias',
    'num_productos'
]

categorical_nominal_features = [
    'genero',
    'region',
    'estado_civil',
    'canal_registro',
    'dia_semana_registro',
    'tiene_tarjeta_credito'
]

categorical_ordinal_features = [
    'uso_app',
    'tipo_plan'
]

##Separacion de variable objetivo y variables predictoras

In [ ]:
X = df[numeric_features + categorical_nominal_features+categorical_ordinal_features]
y = df[target]

Se utilizó LinearRegression como modelo baseline debido a su simplicidad, interpretabilidad y capacidad de establecer una referencia inicial para comparar modelos más complejos.

In [ ]:
# Objetivo Regresion
"""
Imputamos los valores faltantes en la columna original antes de asignarla a y_reg.
Ya que como se mencionó anteriormente, el haber pasado esta columna sin procesar generaria un error
a la hora de evaluar el modelo
"""

#solo eliminar nan
df_reg = df.dropna(subset=['gasto_mensual'])

X_reg = df_reg.drop(columns=['gasto_mensual', 'id_cliente', 'fecha_registro', 'codigo_postal'], errors='ignore')
y_reg = df_reg['gasto_mensual']

#Separacion de variables de entrenamineto y test
X_train, X_test, y_train, y_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=SEED
)



###pipeline regresion lineal

In [ ]:
pipeline_lr = Pipeline(steps=[
    ("duplicados", FunctionTransformer(tratar_duplicados,
                                       kw_args={"drop": False})),
    ('preprocessor', preprocesador),
    ("conversion", DataFrameConverter(preprocesador)),
    ("colinealidad", CorrelationFilter(threshold=0.9)),
    ('model', LinearRegression())
])

El pipeline permite automatizar preprocessing y entrenamiento, asegurando reproducibilidad y evitando data leakage

In [ ]:
pipeline_lr.fit(X_train, y_train)

y_pred = pipeline_lr.predict(X_test)

"""
El modelo intenta predecir 'gasto_mensual'
para los clientes nuevos
"""

"\nEl modelo intenta predecir 'gasto_mensual'\npara los clientes nuevos\n"

###Pipeline arbol de regresion

In [ ]:
pipeline_dtr = Pipeline(steps=[
    ("duplicados", FunctionTransformer(tratar_duplicados, kw_args={"drop": False})),
    ('preprocessor', preprocesador),
    ("conversion", DataFrameConverter(preprocesador)),
    ("colinealidad", CorrelationFilter(threshold=0.9)),
    ('model', DecisionTreeRegressor(random_state=SEED, max_depth=5))
])

In [ ]:
pipeline_dtr.fit(X_train, y_train)

y_pred = pipeline_dtr.predict(X_test)

#Prueba arbol de regresion con hiperparametros

In [ ]:
pipeline_tree = Pipeline([
    ('preprocessor', preprocesador),
    ('model', DecisionTreeRegressor(random_state=42))
])

In [ ]:
param_grid = {
    'model__max_depth': [3, 5, 10, 15, None],
    'model__min_samples_split': [2, 5, 10, 20],
    'model__min_samples_leaf': [1, 2, 5, 10]
}

In [ ]:
grid_search = GridSearchCV(
    estimator=pipeline_tree,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)
"""
Aqui se evalua el 'r2' ya que nuestro proposito
es explicar el fenomemo del gasto mensual de
los clientes, y r2 es precisamente quien nos ayuda
a medir y evaluar esto
"""

"\nAqui se evalua el 'r2' ya que nuestro proposito\nes explicar el fenomemo del gasto mensual de\nlos clientes, y r2 es precisamente quien nos ayuda\na medir y evaluar esto\n"

In [ ]:
print("Mejores hiperparámetros:")
print(grid_search.best_params_)

Mejores hiperparámetros:
{'model__max_depth': 3, 'model__min_samples_leaf': 10, 'model__min_samples_split': 2}


In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
y_pred_best = best_model.predict(X_test)